In [ ]:
# Block 1: Setup, Configuration, and Data Download

!pip install torch torchvision torchaudio scikit-learn opencv-python --quiet

from google.colab import files
import os, random, time
import numpy as np, pandas as pd
import math
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from PIL import Image
import cv2

# --- Data Setup: AUTHENTICATION AND DOWNLOAD ---
COMPETITION_ID = 'nn-26-scene-style-classification'
DATA_DIR = '/content/data'

if not os.path.exists(DATA_DIR):

    if not os.path.exists('kaggle.json'):
        print("Please upload your 'kaggle.json' file when prompted.")
        try:
            files.upload()
        except:
            pass

        # Configure credentials
        !mkdir -p ~/.kaggle
        !mv kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json

    # Download and Unzip (This command failed previously, but is standard for Colab)
    print(f"Attempting to download competition data: {COMPETITION_ID}")
    !kaggle competitions download -c {COMPETITION_ID}

    # Unzip into the data directory
    !unzip -q {COMPETITION_ID}.zip -d {DATA_DIR}
    print("Dataset downloaded and unzipped.")
else:
    print("Dataset directory already exists.")

# Paths
TRAIN_DIR = f'{DATA_DIR}/StyleClassificationIndoors/StyleClassificationIndoors/train'
TEST_DIR  = f'{DATA_DIR}/StyleClassificationIndoors/StyleClassificationIndoors/test'
CLASS_MAPPING = f'{DATA_DIR}/StyleClassificationIndoors/StyleClassificationIndoors/class_mapping.txt'
BEST_MODEL_PATH = 'best_custom_b0_cbam.pth'

# --- Hyperparameters and Overfitting Controls ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

# Core Config
IMG_SIZE = 240
BATCH_SIZE = 16
EPOCHS = 50
LR = 2e-4
NUM_WORKERS = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Overfitting Controls
EARLY_STOP_PATIENCE = 12
print('Device:', DEVICE)

Please upload your 'kaggle.json' file when prompted.


Saving kaggle.json to kaggle.json
Attempting to download competition data: nn-26-scene-style-classification
100% 1.15G/1.15G [00:05<00:00, 226MB/s]
100% 1.15G/1.15G [00:05<00:00, 206MB/s]
Dataset downloaded and unzipped.
Device: cuda


In [ ]:
# Block 2: Data Handling (Split, Normalization, Class Weights, Stronger Augmentation)

# --- Data Transforms (Preprocessing and Stronger Augmentation) ---
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Stronger color jitter and larger crop scale for better generalization
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),

    # Conversion and Normalization
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)

# Split the data (Stratified, 10% validation)
targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))
train_idx, val_idx = train_test_split(indices, test_size=0.1, stratify=targets, random_state=SEED)

full_dataset_val = datasets.ImageFolder(TRAIN_DIR, transform=val_tfms)
val_dataset = Subset(full_dataset_val, val_idx)
train_dataset = Subset(full_dataset, train_idx)

# Fix Unbalanced Data using the Weight Method ---
train_labels = [full_dataset.targets[i] for i in train_idx]
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_w = torch.tensor(class_weights_np, dtype=torch.float32)

print(f'Train size: {len(train_dataset)} | Val size: {len(val_dataset)}')
print(f'Calculated Class Weights (to fix imbalance): {class_w.numpy()}')


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

Train size: 11846 | Val size: 1317
Calculated Class Weights (to fix imbalance): [0.9940421 0.843612  0.9745784 0.9898062 1.0128249 0.9571752 0.9745784
 0.9828259 1.0128249 0.9814416 1.3964399 0.9571752 1.0084277 1.0384852
 1.0026238 1.0026238 1.0202395]


In [ ]:
# Block 3: Custom Model Architecture (EfficientNet-B0 + CBAM from Scratch)

# Activation function
class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

# CBAM Components
class ChannelAttention(nn.Module):
    def __init__(self, in_ch, se_ratio=0.25):
        super().__init__()
        reduced = max(1, int(in_ch * se_ratio))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_ch, reduced, 1); self.act = Swish()
        self.fc2 = nn.Conv2d(reduced, in_ch, 1)
    def forward(self, x):
        y = self.pool(x); y = self.act(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)); return x * y

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1); y = self.conv(y)
        return x * self.sigmoid(y)

class CBAM(nn.Module):
    """Convolutional Block Attention Module (CBAM)"""
    def __init__(self, in_ch, se_ratio=0.25):
        super().__init__(); self.channel_attention = ChannelAttention(in_ch, se_ratio=se_ratio)
        self.spatial_attention = SpatialAttention()
    def forward(self, x):
        x = self.channel_attention(x); x = self.spatial_attention(x)
        return x

class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio, kernel_size, stride, se_ratio=0.25):
        super().__init__()
        self.use_residual = (in_ch == out_ch and stride == 1); mid_ch = in_ch * expand_ratio

        self.expand_conv = nn.Identity() if expand_ratio == 1 else nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 1, bias=False), nn.BatchNorm2d(mid_ch), Swish())
        self.depthwise = nn.Sequential(
            nn.Conv2d(mid_ch, mid_ch, kernel_size, stride=stride, padding=kernel_size//2, groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch), Swish())
        self.attention = CBAM(mid_ch, se_ratio=se_ratio)
        self.project_conv = nn.Sequential(
            nn.Conv2d(mid_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        y = self.expand_conv(x); y = self.depthwise(y); y = self.attention(y); y = self.project_conv(y)
        if self.use_residual: return x + y
        return y

class EfficientNetB0Custom(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), Swish())

        # B0 Configuration
        cfg = [
            (32, 16, 1, 3, 1, 1), (16, 24, 6, 3, 2, 2), (24, 40, 6, 5, 2, 2),
            (40, 80, 6, 3, 2, 3), (80, 112,6, 5, 1, 3), (112,192,6,5,2,4),
            (192,320,6,3,1,1)
        ]

        layers = []
        for in_ch, out_ch, exp, k, s, reps in cfg:
            layers.append(MBConv(in_ch, out_ch, expand_ratio=exp, kernel_size=k, stride=s))
            for _ in range(reps-1):
                layers.append(MBConv(out_ch, out_ch, expand_ratio=exp, kernel_size=k, stride=1))
        self.blocks = nn.Sequential(*layers)

        self.head_conv = nn.Sequential(
            nn.Conv2d(320, 1280, 1, bias=False), nn.BatchNorm2d(1280), Swish())

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x); x = self.blocks(x); x = self.head_conv(x)
        x = self.avgpool(x); x = x.flatten(1); x = self.dropout(x)
        x = self.classifier(x)
        return x

model = EfficientNetB0Custom(num_classes=NUM_CLASSES).to(DEVICE)
print(f'Model loaded. Total parameters: {sum(p.numel() for p in model.parameters()):,}')

Model loaded. Total parameters: 7,166,193


In [ ]:
# Block 4: Optimizer, Loss, and OneCycleLR Setup

# Steps per epoch calculation
STEPS_PER_EPOCH = math.ceil(len(train_dataset) / BATCH_SIZE)
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

# Loss function: Uses class_w (calculated in Block 2) and Label Smoothing (0.1)
criterion = nn.CrossEntropyLoss(
    weight=class_w.to(DEVICE),
    label_smoothing=0.1
)

# Optimizer: AdamW with Weight Decay (1e-3)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

# Scheduler: OneCycleLR for fast, reliable training
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR * 10,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS
)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

print("Loss function, Optimizer, and OneCycleLR Scheduler initialized.")

Loss function, Optimizer, and OneCycleLR Scheduler initialized.


In [ ]:
# Block 5: Training Loop with Early Stopping

# --- Unfreezing Layers (All layers trainable) ---
print("All model layers are UN-FROZEN and trainable to maximize feature learning.")
for param in model.parameters():
    param.requires_grad = True

print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


# --- Training Loop with Early Stopping ---
print(f"\nStarting Training for {EPOCHS} epochs with Early Stopping (Patience={EARLY_STOP_PATIENCE})...")
best_val_acc = 0.0
patience_counter = 0

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0; running_correct = 0; running_total = 0
    t0 = time.time()

    # Training
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        scheduler.step() # Update LR after every batch

        running_loss += loss.item() * imgs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        running_total += labels.size(0)

    train_acc = running_correct / running_total

    # Validation
    model.eval()
    val_loss = 0.0; val_correct = 0; val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    t1 = time.time()
    print(f'Epoch {epoch:02d}/{EPOCHS} | Time: {t1-t0:.1f}s | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')

    # Early Stopping and Saving
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_acc': val_acc}, BEST_MODEL_PATH)
        print('  -> Best model saved.')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'\nEarly stopping triggered after {patience_counter} epochs without improvement.')
            break

print(f'\nTraining finished. Best validation accuracy: {best_val_acc:.4f}')

All model layers are UN-FROZEN and trainable to maximize feature learning.
Total trainable parameters: 7,166,193

Starting Training for 50 epochs with Early Stopping (Patience=12)...
Epoch 01/50 | Time: 176.9s | Train Acc: 0.0919 | Val Acc: 0.1025 | LR: 1.01e-04
  -> Best model saved.
Epoch 02/50 | Time: 145.1s | Train Acc: 0.1392 | Val Acc: 0.1602 | LR: 1.63e-04
  -> Best model saved.
Epoch 03/50 | Time: 145.5s | Train Acc: 0.1626 | Val Acc: 0.1746 | LR: 2.63e-04
  -> Best model saved.
Epoch 04/50 | Time: 146.3s | Train Acc: 0.1721 | Val Acc: 0.1724 | LR: 3.98e-04
Epoch 05/50 | Time: 145.1s | Train Acc: 0.1881 | Val Acc: 0.1891 | LR: 5.60e-04
  -> Best model saved.
Epoch 06/50 | Time: 146.1s | Train Acc: 0.2061 | Val Acc: 0.2202 | LR: 7.43e-04
  -> Best model saved.
Epoch 07/50 | Time: 144.6s | Train Acc: 0.2041 | Val Acc: 0.2058 | LR: 9.40e-04
Epoch 08/50 | Time: 145.2s | Train Acc: 0.2070 | Val Acc: 0.2263 | LR: 1.14e-03
  -> Best model saved.
Epoch 09/50 | Time: 146.9s | Train Acc:

In [ ]:
# Block 6 (MODIFIED): Phase II Fine-Tuning (Slow Decay) with History Collection

import torch
from torch.optim import AdamW
import time
import os

# --- Configuration for Fine-Tuning ---
FT_EPOCHS = 10
FT_LR = 1e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_MODEL_PATH = 'best_custom_b0_cbam.pth'

# Initialize Fine-Tuning History
ft_history = []
print("Fine-tuning history list initialized.")

print(f"\nStarting Phase II Fine-Tuning: {FT_EPOCHS} epochs with fixed LR = {FT_LR:.1e}")

# Load the Best Model Weights from Phase I
if os.path.exists(BEST_MODEL_PATH):
    print("Loading best model weights from initial training phase...")
    ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    best_val_acc = ckpt['val_acc']
else:
    print(f"Error: Best model checkpoint not found at {BEST_MODEL_PATH}. Cannot start fine-tuning.")

# Setup New Optimizer (Fixed Low LR)
ft_optimizer = AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-3)
ft_scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

# Fine-Tuning Loop
patience_counter = 0

for epoch in range(1, FT_EPOCHS + 1):
    model.train()
    running_loss = 0.0; running_correct = 0; running_total = 0
    t0 = time.time()

    # Training
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
        ft_optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        ft_scaler.scale(loss).backward()
        ft_scaler.step(ft_optimizer); ft_scaler.update()

        running_loss += loss.item() * imgs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        running_total += labels.size(0)

    train_acc = running_correct / running_total

    # Validation
    model.eval()
    val_loss = 0.0; val_correct = 0; val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    t1 = time.time()
    print(f'FT Epoch {epoch:02d}/{FT_EPOCHS} | Time: {t1-t0:.1f}s | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {FT_LR:.1e}')

    ft_history.append({
        'train_acc': train_acc,
        'val_acc': val_acc,
        'train_loss': running_loss / running_total,
        'val_loss': val_loss / val_total,
        'lr': FT_LR
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_acc': val_acc}, BEST_MODEL_PATH)
        print('  -> Best fine-tuned model saved.')
    else:
        patience_counter += 1
        if patience_counter >= 3:
            print(f'\nEarly stopping triggered after {patience_counter} epochs without improvement during fine-tuning.')
            break

print(f'\nPhase II Fine-Tuning finished. Final best validation accuracy: {best_val_acc:.4f}')

Fine-tuning history list initialized.

Starting Phase II Fine-Tuning: 10 epochs with fixed LR = 1.0e-05
Loading best model weights from initial training phase...
FT Epoch 01/10 | Time: 147.5s | Train Acc: 0.5146 | Val Acc: 0.3311 | LR: 1.0e-05
FT Epoch 02/10 | Time: 145.2s | Train Acc: 0.5149 | Val Acc: 0.3349 | LR: 1.0e-05
FT Epoch 03/10 | Time: 148.4s | Train Acc: 0.5183 | Val Acc: 0.3432 | LR: 1.0e-05

Early stopping triggered after 3 epochs without improvement during fine-tuning.

Phase II Fine-Tuning finished. Final best validation accuracy: 0.3432


In [ ]:
# Block 9: Plot Fine-Tuning History

import matplotlib.pyplot as plt
from google.colab import files

def plot_ft_history(ft_history):
    """Generates and saves plots for the Fine-Tuning history."""
    if not ft_history:
        print("Fine-Tuning history list is empty. Cannot plot results.")
        return

    # Extract metrics
    epochs = range(1, len(ft_history) + 1)
    train_acc = [h['train_acc'] for h in ft_history]
    val_acc = [h['val_acc'] for h in ft_history]
    train_loss = [h['train_loss'] for h in ft_history]
    val_loss = [h['val_loss'] for h in ft_history]

    plt.style.use('ggplot')

    # 1. Accuracy Plot
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_acc, 'bo-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'ro-', label='Validation Accuracy')
    plt.title('Fine-Tuning Accuracy (Phase II)')
    plt.xlabel('FT Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # 2. Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_loss, 'bo-', label='Training Loss')
    plt.plot(epochs, val_loss, 'ro-', label='Validation Loss')
    plt.title('Fine-Tuning Loss (Phase II)')
    plt.xlabel('FT Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plot_filename = 'finetuning_metrics.png'
    plt.savefig(plot_filename)
    print(f"Fine-Tuning plots saved as {plot_filename}")

    plt.close('all')

    files.download(plot_filename)

if 'ft_history' in locals() and ft_history:
    plot_ft_history(ft_history)
else:
    print("The 'ft_history' list variable was not found or is empty. Please ensure you ran the MODIFIED Block 7 first.")

Fine-Tuning plots saved as finetuning_metrics.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Block 10 Prediction, CSV Output, and Model Save

import torch
import os
import pandas as pd
from PIL import Image
import cv2
from google.colab import files

print("\n--- Generating Final Test Predictions using the BEST SAVED MODEL ---")

# --- Configuration (Ensuring all variables are available) ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_MODEL_PATH = 'best_custom_b0_cbam.pth' # The best checkpoint from Block 5/7
FINAL_MODEL_FILE = 'final_submission_model.pt' # New file for the final, full model object

# Load best checkpoint
if not os.path.exists(BEST_MODEL_PATH):
    print(f"FATAL ERROR: Best model checkpoint not found at {BEST_MODEL_PATH}.")
    print("Please ensure Block 5 and Block 7 ran successfully and saved the model.")

ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f"Model loaded successfully. Best validation accuracy recorded: {ckpt['val_acc']:.4f}")


# Helper function for robust inference image loading
def load_image_for_inference(p):
    """Loads and preprocesses image, handling potential corruption/errors."""
    try:
        return Image.open(p).convert('RGB')
    except:
        try:
            # Fallback using OpenCV for robust loading
            arr = cv2.imread(p)
            if arr is None: return None
            arr = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
            return Image.fromarray(arr)
        except:
            return None

# --- Load Class Mappings ---
mapping = {}
if os.path.exists(CLASS_MAPPING):
    with open(CLASS_MAPPING, 'r') as f:
        for line in f:
            if ':' in line:
                name, idx = line.strip().split(':')
                mapping[name.strip()] = int(idx.strip())

# Get all test image files
test_paths = sorted([p for p in os.listdir(TEST_DIR) if p.lower().endswith(('.jpg','.png','.jpeg'))])
print(f'Test images found: {len(test_paths)}')

rows=[]
with torch.no_grad():
    for fname in test_paths:
        p = os.path.join(TEST_DIR, fname)
        img_pil = load_image_for_inference(p)

        if img_pil is None:
            rows.append((fname, -1)); continue

        # Apply validation transform and prepare for model
        img_t = val_tfms(img_pil).unsqueeze(0).to(DEVICE)

        # Inference
        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            out = model(img_t)

        cls_idx = int(out.argmax(1).item())
        class_name = class_names[cls_idx]

        # Map the class name back to the competition's required numerical label
        mapped = mapping.get(class_name, cls_idx)
        rows.append((fname, int(mapped)))

# --- Create and Save Submission CSV ---
submission = pd.DataFrame(rows, columns=['ImageName','ClassLabel'])
submission.to_csv('submission.csv', index=False)

print('\nTest Predictions saved to submission.csv')
print('Sample Submission:')
print(submission.head())

try:
    # Saves the entire model
    torch.save(model, FINAL_MODEL_FILE)
    print(f"\n Final Model saved successfully as {FINAL_MODEL_FILE}")
    files.download(FINAL_MODEL_FILE)
except Exception as e:
    print(f"Error saving final model: {e}")

# Download the final CSV file
files.download('submission.csv')


--- Generating Final Test Predictions using the BEST SAVED MODEL ---
Model loaded successfully. Best validation accuracy recorded: 0.3432
Test images found: 5482

Test Predictions saved to submission.csv
Sample Submission:
            ImageName  ClassLabel
0     testimage_1.jpg           9
1    testimage_10.jpg          14
2   testimage_100.jpg          10
3  testimage_1000.jpg          12
4  testimage_1001.jpg          10

✅ Final Model saved successfully as final_submission_model.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>